# Parabolic Problem

In [0]:
%%capture
!apt-get install -y -qq software-properties-common python-software-properties module-init-tools
!add-apt-repository -y ppa:fenics-packages/fenics
!apt-get update -qq
!apt install -y --no-install-recommends fenics
!rm -rf *
from fenics import *
import matplotlib.pyplot as plt
import numpy as np
from tabulate import tabulate

On the unit square domain $\Omega = (0,1) \times (0,1)$ and time interval $I = (0, 1)$ 
  we consider the initial and boundary value problem
  $$
    \begin{cases}
      \dfrac{\partial u}{\partial t}- \Delta u = f  & \text{in } \Omega\times I, \\
      u = g                                         & \text{on } \partial\Omega\times I, \\
      {\left. u \right|}_{t=0} = u_0                & \text{on } \Omega. \\
    \end{cases}
$$

As exact solution we consider $u(x, y, t) = 1 + \frac{1}{2} \sin(\omega t) \sin(\pi x) \sin(\pi y)$. 
Hence, we have $u_0=0$, $g(x, y, t)=1$ and 
$$
f(x, y, t)= \frac12 \left(\omega \cos(\omega t) + 2 \pi^2 \sin(\omega t)\right) \sin(\pi x)\sin(\pi y). 
$$
We take $\omega=\pi$ and consider different values of $\theta$ and $\delta t$.

## Weak formulation
\begin{equation}
\int_\Omega \dfrac{\partial u}{\partial t}v \space d \Omega 
- \int_\Omega \Delta u v \space d \Omega
= \int_\Omega fv \space d \Omega
\end{equation}

Sostituendo $ \begin{equation} u=u-g \end{equation} $:
\begin{equation}
\int_\Omega \dfrac{\partial u}{\partial t}v \space d \Omega 
+ \int_\Omega \nabla u \nabla v \space d \Omega
= \int_\Omega fv \space d \Omega + \int_\Omega \nabla g \nabla v \space d \Omega
\end{equation}

Theta-Method
\begin{equation}
\dfrac{\partial (u^{n+1}, v)}{\partial t}
+ \theta a(u^{n+1}, v) = \theta(f^{n+1}, v) + (1-\theta)(f^{n}, v) - (1-\theta) a(u^{n}, v) + a(u_g, v)+\dfrac{\partial (u^{n}, v)}{\partial t}
\end{equation}

In [0]:
# Mesh e spazio
def Parabolic(n, u_ex, f, g, deg, theta, dt):
  mesh = UnitSquareMesh(n, n, 'crossed')
  V = FunctionSpace(mesh, 'CG', deg)
  u_old = project(u_ex, V)
  
  def boundary(x, on_boundary):
    return on_boundary

  bc = DirichletBC(V, g, boundary)

  u = TrialFunction(V)
  v = TestFunction(V)

  a = (u * v / dt + theta * inner(grad(u), grad(v))) * dx
  L1 = f*v*dx
  L2 = (-(1-theta)*inner(grad(u_old), grad(v)) + (u_old*v)/dt)*dx
  A = assemble(a)
  bc.apply(A)
  Ainv=LUSolver(A)
  return Ainv, A, L1, L2, mesh, bc, u_old

deg = 1
g = Constant(1.0)
Tfin = 1
for n in [5, 10, 20, 40]:
  print('n=' + str(n))
  for dt in [0.1, 0.01, 0.0001]:

    u_ex = Expression('1 + 0.5 * sin(omega * time) * sin(pi * x[0]) * sin( pi * x[1])', degree=2*deg+1, time=0.0, omega=pi)
    f = Expression('0.5 * (omega * cos(omega * time) + 2 * pi * pi * sin(omega * time)) * sin(pi * x[0]) * sin(pi * x[1])', degree=deg+1, time=0.0, omega=pi)

    theta_vect = [0, 1, 0.5]
    l2errvect=[]
    h1errvect=[]

    for theta in theta_vect:
      Ainv, A, L1, L2, mesh, bc, u_old = Parabolic (n, u_ex, f, g, deg, theta, dt)

      l2err = errornorm(u_ex, u_old, 'L2')
      h1err = errornorm(u_ex, u_old, 'H10')
      t = 0
      while t < Tfin:
        b1=L1

        t=t+dt
        f.time=t
        b2=L1
        b3=L2

        b=(1-theta)*b1+theta*b2+b3
        b = assemble(b)
        bc.apply(b)
      #  u = solve(A, x, b)
        Ainv.solve(u_old.vector(), b)
        u_ex.time=t

        l2err = max(l2err, errornorm(u_ex, u_old, 'L2'))
        h1err = max(h1err, errornorm(u_ex, u_old, 'H10'))
      l2errvect.append(l2err)
      h1errvect.append(h1err)

    print('dt='+ str(dt))
    print(tabulate([[theta_vect[0], l2errvect[0], h1errvect[0]], [theta_vect[1], l2errvect[1], h1errvect[1]], [theta_vect[2], l2errvect[2], h1errvect[2]]], headers=['Theta', 'L2 error', 'H10 error']))
    print()
    print()
  
  print()
  print()
  print()
  print()  

n=5
dt=0.1
  Theta     L2 error    H10 error
-------  -----------  -----------
    0    8.43468e+18  3.39571e+20
    1    0.0277882    0.185529
    0.5  0.0394536    0.183953


dt=0.01
  Theta      L2 error    H10 error
-------  ------------  -----------
    0    2.94519e+112  1.1857e+114
    1    0.0082154     0.183744
    0.5  0.00882922    0.183733


dt=0.0001
  Theta    L2 error    H10 error
-------  ----------  -----------
    0    0.00770356     0.183733
    1    0.00770541     0.183733
    0.5  0.00770441     0.183733






n=10
dt=0.1
  Theta     L2 error    H10 error
-------  -----------  -----------
    0    4.98017e+24  4.16893e+26
    1    0.0270689    0.121679
    0.5  0.039269     0.175134


dt=0.01
  Theta      L2 error    H10 error
-------  ------------  -----------
    0    inf           inf
    1      0.00246619    0.091966
    0.5    0.00449875    0.0919322


dt=0.0001
  Theta    L2 error    H10 error
-------  ----------  -----------
    0    0.00191787    0.0919323


# Thermal Problem

$$
    \frac{\partial u}{\partial t} - k \Delta u = 0 \quad \text{in} \; \Omega \;\text{ and } \; t>0,
$$
with $ \Omega= (0,1) \times (0,1)$

The following BC are applied:


1.   $
  u = 40 \quad\text{ on }  
  \Gamma_D = \left\{(x,y)\in \partial\Omega \,|\, x = 0 \text{ and } y \in \left(\frac{1}{4}, \frac{3}{4}\right) \right\},
$
2. $-k \nabla u \cdot \vec{n} = \alpha (u-u_{\text{env}}),\quad\text{ on } 
  \Gamma_R= \left\{(x,y) \in \partial\Omega \,|\, x \in \left(\frac{1}{4}, \frac{3}{4}\right) \text{ and } y = 1\right\}$ 
  $$u_{env}=5$$
3.$-k \nabla u \cdot \vec{n} = 0 $ on the rest of the boundary.

In [0]:
# Data & Mesh
n=50
mesh = UnitSquareMesh(n, n, 'crossed')
deg=1
V= FunctionSpace(mesh, 'CG', deg)
u_env = Constant(5.0)
g = Constant(40)
k = 1.e-2
alpha = 1.e-2
u_old = project(Constant(20.0), V)
dt=1
Tfin=200.0
vtkfile = File('Lab04_thermalProblem.pvd')

In [0]:
# Boundary Conditions
dim = mesh.geometric_dimension()

boundary_markers = MeshFunction('size_t', mesh, dim-1, 0)

class Boundary(SubDomain):
  def inside(self, x, on_boundary):
    return on_boundary

class DirichletBoundary(SubDomain):
  def inside(self, x, on_boundary):
    return on_boundary and near(x[0], 0.0) and (0.25<=x[1]<=0.75)

class RobinBoundary(SubDomain):
  def inside(self, x, on_boundary):
    return on_boundary and near(x[1], 0.0) and (0.25<=x[0]<=0.75)

dx = Measure('dx', domain=mesh)
ds = Measure('ds', domain=mesh, subdomain_data=boundary_markers)

Boundary().mark(boundary_markers, 1)
DirichletBoundary().mark(boundary_markers, 2)
RobinBoundary().mark(boundary_markers, 3)

bc = DirichletBC(V, g, boundary_markers, 2) # strong enforcement of Dirichlet BC (on V)

In [0]:
# Assembly
u=TrialFunction(V)
v=TestFunction(V)

a = (u*v/dt+k*inner(grad(u), grad(v)))*dx + (alpha*u*v)*ds(3)
L = (u_old*v/dt)*dx -(alpha*u_env*v)*ds(3)

A = assemble(a)
bc.apply(A)
Ainv=LUSolver(A)

t=0
vtkfile << (u_old, t)
while t<=Tfin:

  t += dt
  b = assemble(L)
  bc.apply(b)

  Ainv.solve(u_old.vector(), b)
  vtkfile << (u_old, t)

In [0]:
!zip -r /content/Solution.zip /content/

updating: content/ (stored 0%)
updating: content/.config/ (stored 0%)
updating: content/.config/logs/ (stored 0%)
updating: content/.config/logs/2020.04.03/ (stored 0%)
updating: content/.config/logs/2020.04.03/16.24.09.722063.log (deflated 85%)
updating: content/.config/logs/2020.04.03/16.23.57.403439.log (deflated 53%)
updating: content/.config/logs/2020.04.03/16.24.26.990500.log (deflated 53%)
updating: content/.config/logs/2020.04.03/16.23.40.043713.log (deflated 91%)
updating: content/.config/logs/2020.04.03/16.24.26.483358.log (deflated 54%)
updating: content/.config/logs/2020.04.03/16.24.13.655529.log (deflated 54%)
updating: content/.config/.last_survey_prompt.yaml (stored 0%)
updating: content/.config/.last_update_check.json (deflated 22%)
updating: content/.config/configurations/ (stored 0%)
updating: content/.config/configurations/config_default (deflated 15%)
updating: content/.config/active_config (stored 0%)
updating: content/.config/.last_opt_in_prompt.yaml (stored 0%)
u